# Этап 01. Базовые данные SberIndex

Берём три исходных файла SberIndex: расходы, доступность рынков и транспортные связи.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with (ROOT / 'configs' / 'sber.json').open(encoding='utf-8') as file:
    config = json.load(file)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


In [2]:
from src.sber import load_sber, build_sber_views, save_sber_views, SHARE_COLUMNS


## Загрузка


In [4]:
consumption, market_access, connection = load_sber(config, ROOT)

print('consumption:', consumption.shape)
print('market_access:', market_access.shape)
print('connection:', connection.shape)


consumption: (303126, 4)
market_access: (2571, 2)
connection: (5942364, 4)


## Матрицы и транспортный граф


In [5]:
views = build_sber_views(
    consumption,
    market_access,
    connection,
    config['calendar'],
)

pd.Series(views['diagnostics']).to_frame('value')


,value
months,24.000
municipalities_with_spending,"2,190.000"
municipalities_complete,"2,016.000"
consumer_rows,"48,384.000"
consumer_features,6.000
market_access_coverage,0.994
transport_edges,"2,800,078.000"


In [6]:
X_consumer = views['X_consumer']
X_market_access = views['X_market_access']
X_sber_context = views['X_sber_context']
transport_edges = views['transport_edges']

assert X_consumer.index.is_unique
assert np.isfinite(X_consumer).all().all()
assert np.allclose(X_consumer[SHARE_COLUMNS].sum(axis=1), 1.0)

print('X_consumer:', X_consumer.shape)
print('X_market_access:', X_market_access.shape)
print('transport_edges:', transport_edges.shape)
display(X_consumer.head())


X_consumer: (48384, 6)
X_market_access: (2016, 1)
transport_edges: (2800078, 5)


share_food  share_health  share_marketplace  \
territory_id date                                                      
1            2023-01-01       0.356         0.059              0.116   
             2023-02-01       0.364         0.056              0.121   
             2023-03-01       0.356         0.057              0.122   
             2023-04-01       0.369         0.055              0.122   
             2023-05-01       0.365         0.054              0.125   

                         share_dining  share_transport  share_other  
territory_id date                                                    
1            2023-01-01         0.053            0.080        0.336  
             2023-02-01         0.048            0.075        0.336  
             2023-03-01         0.050            0.074        0.341  
             2023-04-01         0.050            0.076        0.328  
             2023-05-01         0.055            0.076        0.325

`X_consumer` — базовая матрица проекта. Её индекс — `territory_id × date`, а признаки:

`share_food`, `share_health`, `share_marketplace`, `share_dining`, `share_transport`, `share_other`.



`market_access` сохраняется отдельным статическим view. Он не добавляется автоматически к потребительским долям. `connection` остаётся списком рёбер для транспортного графа. 


In [7]:
output_dir = ROOT / config['output_dir']
paths = save_sber_views(views, output_dir)

for name, path in paths.items():
    print(f'{name:18s} {path.relative_to(ROOT)}')


X_consumer         outputs/base/X_consumer.parquet
X_market_access    outputs/base/X_market_access.parquet
X_sber_context     outputs/base/X_sber_context.parquet
transport_edges    outputs/base/transport_edges.parquet
